In [1]:
!pip install pandas scikit-learn streamlit requests

In [1]:
import pandas as pd
import ast # We will need this later to clean up the text

In [4]:
# Load the correct movies file
movies = pd.read_csv('tmdb_5000_movies.csv')

# Keep only the essential columns for a simple recommender
movies = movies[['id', 'title', 'overview', 'genres']]

# Drop any rows with missing data to prevent errors
movies.dropna(inplace=True)

# Display the first 3 rows to confirm it worked
movies.head(3)

,id,title,overview,genres
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."


In [5]:
import ast

# 1. Function to extract just the genre names
def clean_genres(text):
    genre_list = []
    for i in ast.literal_eval(text):
        genre_list.append(i['name'])
    return " ".join(genre_list)

# 2. Apply it to clean the column
movies['genres'] = movies['genres'].apply(clean_genres)

# 3. Create a single "tags" column combining overview and genres
movies['tags'] = movies['overview'] + " " + movies['genres']

# 4. Create our final, clean dataframe
new_df = movies[['id', 'title', 'tags']].copy()

# Make all text lowercase to avoid duplication
new_df['tags'] = new_df['tags'].str.lower()
new_df.head(3)

,id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Convert text into a matrix of numbers (ignoring common words like "the" or "and")
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

# 2. Calculate the distance (similarity) between every single movie
similarity = cosine_similarity(vectors)

print("Machine Learning math complete! Matrix shape:", similarity.shape)

Machine Learning math complete! Matrix shape: (4800, 4800)


In [7]:
def recommend(movie):
    # Find the row number of the movie the user typed in
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    # Get the similarity scores for this specific movie
    distances = similarity[movie_index]
    
    # Sort the movies to find the top 5 highest scores
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # Print the titles of the recommended movies
    print(f"Because you watched {movie}, we recommend:")
    print("-" * 30)
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

# Let's test it!
recommend('Avatar')

Because you watched Avatar, we recommend:
------------------------------
Beowulf
The Helix... Loaded
Small Soldiers
Iron Man 3
Journey 2: The Mysterious Island


In [8]:
import pickle

# Save the movie list as a dictionary (safer for web apps)
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Save the math/distance matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Files saved! Check your folder for the two new .pkl files.")

Files saved! Check your folder for the two new .pkl files.


In [9]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd

# 1. UI Setup: Wide layout and dark mode styling
st.set_page_config(page_title="Movie Recommender", layout="wide")

st.markdown("""
<style>
    /* Adding a subtle card effect for the recommendations */
    .movie-title {
        font-size: 18px;
        font-weight: bold;
        text-align: center;
        margin-top: 10px;
        color: #E0E0E0;
    }
</style>
""", unsafe_allow_html=True)

st.title("🎥 Cinematic Matchmaker")
st.write("Select a movie you love, and the algorithm will find your next favorite.")

# 2. Load the exported "brain"
movies_dict = pickle.load(open('movie_dict.pkl', 'rb'))
movies = pd.DataFrame(movies_dict)
similarity = pickle.load(open('similarity.pkl', 'rb'))

# 3. The Recommendation Logic for the Web App
def recommend(movie):
    movie_index = movies[movies['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    recommended_movies = []
    for i in movies_list:
        recommended_movies.append(movies.iloc[i[0]].title)
    return recommended_movies

# 4. The Interactive Interface
selected_movie = st.selectbox(
    "Search for a movie:",
    movies['title'].values
)

if st.button('Recommend'):
    recommendations = recommend(selected_movie)
    
    # 5. Displaying the UI in a 5-column grid
    cols = st.columns(5)
    
    for idx, col in enumerate(cols):
        with col:
            # Using a sleek placeholder image for now to keep the layout structured
            st.image("https://via.placeholder.com/300x450/262730/FFFFFF?text=Poster+Coming+Soon")
            st.markdown(f"<div class='movie-title'>{recommendations[idx]}</div>", unsafe_allow_html=True)

Writing app.py
